# Explainable ML for DDoS detection (CSE-CIC-IDS2018): XGBoost + SHAP

COMP6310 group project. Single supervised model (XGBoost) that outputs a flag (benign / DDoS) and a model score from 0 to 1. The score is the model's raw output and has not been calibrated (training uses `scale_pos_weight`), so it is not a probability.

**Data:** processed CICFlowMeter CSVs for 20 and 21 Feb 2018 (LOIC-HTTP on the 20th; HOIC and LOIC-UDP on the 21st).

**Things this notebook already accounts for (found while running it):**
- The Wednesday file has **no `Dst IP`**, so feature 7 (flows to the same destination in a 10 s window) cannot be built for HOIC / LOIC-UDP. It is switched off.
- LOIC-UDP has only 1,730 flows, so any LOIC-UDP result is noisy.
- Tool and day are tied together (LOIC-HTTP = Tuesday only; HOIC and LOIC-UDP = Wednesday only).

## 0. Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)

## 1. Settings (confirmed against the real files)

In [ ]:
DATA_DIR = Path("../data")      # put the CSV files here; do not commit them to GitHub
FILES = ["Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
         "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"]

LABEL, TIME, DST = "Label", "Timestamp", "Dst IP"
BASE_FEATURES = [
    "Flow Pkts/s", "Flow Byts/s", "Flow Duration",
    "Flow IAT Mean", "Flow IAT Std", "SYN Flag Cnt",
    "Fwd Pkt Len Mean", "Fwd Pkt Len Std",
]
COUNT_COLS = ["Tot Fwd Pkts", "Tot Bwd Pkts"]   # used to build the fwd/bwd ratio feature
# Candidate extra columns, only tested in section 9c (NOT part of the base model inputs).
# Rows are never dropped because of these; missing values are left for XGBoost to handle.
# Update 3: raw columns for candidate features 12-15 (tested in 9b/9c only, not model inputs yet).
CANDIDATE_COLS = ["TotLen Fwd Pkts", "TotLen Bwd Pkts", "Fwd IAT Mean", "Fwd IAT Min",
                  "Active Mean", "Idle Mean", "PSH Flag Cnt", "Fwd Act Data Pkts"]
EXTRA_COLS = ["Bwd Pkt Len Mean", "RST Flag Cnt", "FIN Flag Cnt"] + CANDIDATE_COLS
USE_WINDOW_FEATURE = False   # feature 7 OFF: Wednesday file has no Dst IP
WINDOW = "10s"               # only used if USE_WINDOW_FEATURE is True
HELD_OUT_TOOL = "HOIC"       # one of: LOIC-HTTP, LOIC-UDP, HOIC

## 2. Load the data

The files are large, so only the needed columns are read. Some copies contain repeated header rows, so rows where the label equals "Label" are removed and numeric columns are forced to numbers.

In [ ]:
def load(path):
    head = pd.read_csv(path, nrows=0).columns.tolist()
    print(path.name, "-", len(head), "columns | has Dst IP:", DST in head)
    for need in [LABEL, TIME] + BASE_FEATURES + COUNT_COLS:
        if need not in head:
            print("   MISSING column:", need)
    for opt in EXTRA_COLS:
        if opt not in head:
            print("   optional column missing (skipped):", opt)
    wanted = [LABEL, TIME] + BASE_FEATURES + COUNT_COLS + EXTRA_COLS + ([DST] if USE_WINDOW_FEATURE else [])
    keep = [c for c in head if c in set(wanted)]
    df = pd.read_csv(path, usecols=keep, low_memory=False)
    df = df[df[LABEL] != LABEL].copy()
    for c in BASE_FEATURES + COUNT_COLS + EXTRA_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if TIME in df:
        df[TIME] = pd.to_datetime(df[TIME], dayfirst=True, errors="coerce")
    df["source_file"] = path.name
    return df

df = pd.concat([load(DATA_DIR / f) for f in FILES], ignore_index=True)
print(df.shape)

## 3. Inspect the labels

In [ ]:
print(df[LABEL].value_counts())
print(df.groupby("source_file")[LABEL].value_counts())
print("timestamp non-null share per file:")
print(df.groupby("source_file")[TIME].apply(lambda s: s.notna().mean()))

## 4. Keep benign and DDoS flows, and name the tool

In [ ]:
def tool_of(label):
    s = str(label).lower()
    if "benign" in s: return "Benign"
    if "udp" in s:    return "LOIC-UDP"
    if "hoic" in s:   return "HOIC"
    if "http" in s:   return "LOIC-HTTP"
    return "Other"

df["tool"] = df[LABEL].map(tool_of)
df = df[df["tool"] != "Other"].copy()
df["y"] = (df["tool"] != "Benign").astype(int)     # 1 = DDoS (positive class)
print(df["tool"].value_counts())

## 5. Clean (infinite, missing and duplicate values)

In [ ]:
n0 = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=BASE_FEATURES + COUNT_COLS)
# duplicates are judged on the original columns only, so row counts match earlier runs
df = df.drop_duplicates(subset=[c for c in df.columns if c not in EXTRA_COLS + ["source_file"]])
print(f"rows: {n0} -> {len(df)}")
print(df["tool"].value_counts())

## 6. Build the features

Derived columns: `fwd_bwd_ratio` (feature 3), `no_reply` and `empty_payload` (feature 10) and `iat_cv` (feature 11). Identifiers (IP addresses, timestamps) are never model inputs.

Model inputs with the window feature off (feature 7 dropped): the original **9 columns** (`FEATURES_BASE9`) plus the four added groups, feature 8 `Bwd Pkt Len Mean`, feature 9 `RST Flag Cnt` and `FIN Flag Cnt`, feature 10 `no_reply` and `empty_payload`, feature 11 `iat_cv`: **15 columns** in `FEATURES`. A group is left out automatically if its source column is missing from the files.

**Update 3 candidates (not model inputs):** feature 12 `bwd_fwd_bytes_ratio` = TotLen Bwd Pkts / (TotLen Fwd Pkts + 1); feature 13 `Fwd IAT Mean`, `Fwd IAT Min`; feature 14 `Active Mean`, `Idle Mean`; feature 15 `PSH Flag Cnt` and `data_share` = Fwd Act Data Pkts / Tot Fwd Pkts. They are built here and tested in sections 9b and 9c; `FEATURES` is unchanged.

In [ ]:
df["fwd_bwd_ratio"] = df["Tot Fwd Pkts"] / (df["Tot Bwd Pkts"] + 1)                     # feature 3
df["no_reply"] = (df["Tot Bwd Pkts"] == 0).astype(int)                                  # feature 10
df["empty_payload"] = (df["Fwd Pkt Len Mean"] == 0).astype(int)                         # feature 10
df["iat_cv"] = (df["Flow IAT Std"] / (df["Flow IAT Mean"] + 1)).replace([np.inf, -np.inf], np.nan)  # feature 11

FEATURES_BASE9 = BASE_FEATURES + ["fwd_bwd_ratio"]        # the original nine columns
NEW_GROUPS = {                                            # features 8 to 11, added after the first experiments
    "reply":      [c for c in ["Bwd Pkt Len Mean"] if c in df],            # feature 8
    "flags":      [c for c in ["RST Flag Cnt", "FIN Flag Cnt"] if c in df],  # feature 9
    "indicators": ["no_reply", "empty_payload"],                           # feature 10
    "timing":     ["iat_cv"],                                              # feature 11
}
FEATURES = FEATURES_BASE9 + [c for cols in NEW_GROUPS.values() for c in cols]

# Update 3 candidates (features 12-15). Chosen for their security meaning; NOT model inputs.
# Sections 9b and 9c test them; FEATURES above is unchanged.
if {"TotLen Bwd Pkts", "TotLen Fwd Pkts"} <= set(df):
    df["bwd_fwd_bytes_ratio"] = df["TotLen Bwd Pkts"] / (df["TotLen Fwd Pkts"] + 1)            # feature 12
if "Fwd Act Data Pkts" in df:
    df["data_share"] = (df["Fwd Act Data Pkts"] / df["Tot Fwd Pkts"]).replace([np.inf, -np.inf], np.nan)  # feature 15
CANDIDATE_GROUPS = {
    "response_ratio":  [c for c in ["bwd_fwd_bytes_ratio"] if c in df],          # feature 12
    "sender_timing":   [c for c in ["Fwd IAT Mean", "Fwd IAT Min"] if c in df],  # feature 13
    "activity":        [c for c in ["Active Mean", "Idle Mean"] if c in df],     # feature 14
    "payload_signals": [c for c in ["PSH Flag Cnt", "data_share"] if c in df],   # feature 15
}
CANDIDATES = [c for cols in CANDIDATE_GROUPS.values() for c in cols]
print("Candidate columns (update 3, not model inputs):", CANDIDATES)

if USE_WINDOW_FEATURE and DST in df and df[DST].notna().all() and TIME in df:
    df["_win"] = df[TIME].dt.floor(WINDOW)
    df["flows_to_dst"] = df.groupby([DST, "_win"])[DST].transform("size")
    FEATURES.append("flows_to_dst")
else:
    print("Window feature skipped (destination IP not available for every file).")
print("Model inputs:", FEATURES)

## 7. Held-out tool experiment

Train on the other tools, test on the held-out tool. Two ways to split the benign flows:

- `per_day`: within each day's file, the earlier 70% of benign flows go to training and the later 30% to testing.
- `by_file`: benign flows of the earlier day (Tuesday) train, the later day (Wednesday) test. Only meaningful when HOIC is held out, because HOIC is entirely on Wednesday.

Training benign flows are capped at 1,000,000 (random sample) for speed. The test set uses all held-out benign flows. Class imbalance is handled with `scale_pos_weight` (no SMOTE yet).

In [ ]:
def run(held_out, benign_mode="per_day", train_frac=0.7, benign_cap=1_000_000, seed=0):
    d = df[df[TIME].notna()]
    ben = d[d["tool"] == "Benign"]
    att = d[d["tool"] != "Benign"]

    if benign_mode == "per_day":
        pct = ben.groupby("source_file")[TIME].rank(pct=True, method="first")
        ben_tr, ben_te = ben[pct <= train_frac], ben[pct > train_frac]
    else:  # "by_file"
        order = ben.groupby("source_file")[TIME].min().sort_values().index
        ben_tr = ben[ben["source_file"] == order[0]]
        ben_te = ben[ben["source_file"] == order[1]]

    tr = pd.concat([ben_tr.sample(min(len(ben_tr), benign_cap), random_state=seed),
                    att[att["tool"] != held_out]])
    te = pd.concat([ben_te, att[att["tool"] == held_out]])

    spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
    model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                          tree_method="hist", scale_pos_weight=spw,
                          n_jobs=-1, random_state=seed)
    model.fit(tr[FEATURES], tr["y"])

    p = model.predict_proba(te[FEATURES])[:, 1]
    pred, y = p >= 0.5, te["y"].values
    print(f"held out {held_out} | benign split: {benign_mode}")
    print(f"  train: {(tr.y==0).sum():,} benign, {(tr.y==1).sum():,} attack")
    print(f"  test:  {(y==0).sum():,} benign, {(y==1).sum():,} attack")
    print(f"  recall on {held_out}: {pred[y==1].mean():.3f}")
    print(f"  false positive rate on benign: {pred[y==0].mean():.4f}  ({int(pred[y==0].sum()):,} false alarms)")
    print(f"  PR-AUC: {average_precision_score(y, p):.3f}  (base rate {y.mean():.3f})")
    return model, te

m1, te1 = run("HOIC", "per_day")
m2, te2 = run("HOIC", "by_file")
m3, te3 = run("LOIC-HTTP", "per_day")

## 8. Diagnostics: score distribution, thresholds, feature importance

Read the AUC numbers with care. When most attack scores sit near 0, a very high ROC-AUC or PR-AUC can come from tiny differences in the ranking, so report **recall at the fixed 0.5 threshold** as the main number.

In [ ]:
def diagnose(name, model, te):
    p = model.predict_proba(te[FEATURES])[:, 1]
    y = te["y"].values
    print(name, "| ROC-AUC:", round(roc_auc_score(y, p), 4))
    print("  attack score 10/50/90th pct:", np.percentile(p[y == 1], [10, 50, 90]).round(3))
    print("  benign score 50/99/99.9th pct:", np.percentile(p[y == 0], [50, 99, 99.9]).round(4))
    for t in [0.5, 0.2, 0.1, 0.05, 0.01]:
        pr = p >= t
        print(f"  threshold {t}: recall {pr[y==1].mean():.3f}, FPR {pr[y==0].mean():.4f}")

diagnose("HOIC held out, per_day", m1, te1)
diagnose("HOIC held out, by_file", m2, te2)
diagnose("LOIC-HTTP held out, per_day", m3, te3)

print(te2.groupby("tool")[FEATURES].median().T)
print({k: round(v, 1) for k, v in m2.get_booster().get_score(importance_type="gain").items()})

## 9. Profile of each attack tool

In [ ]:
att = df[df["tool"] != "Benign"]
print((att["Fwd Pkt Len Mean"] == 0).groupby(att["tool"]).mean().rename("share with zero fwd payload"))
print(att.groupby("tool")[FEATURES].median().T.round(1))
print((att["SYN Flag Cnt"] != 0).groupby(att["tool"]).mean().rename("share with non-zero SYN Flag Cnt"))

## 9b. Feature evidence: each attack tool vs benign flows from the SAME file

Use this to check every assumption in the Week 3 feature table against the data. Update 3: it also covers the candidate features 12-15. Comparing against benign flows from the same day avoids the day confound (LOIC-HTTP is Tuesday only; HOIC and LOIC-UDP are Wednesday only).

- `direction`: whether the attack median is higher or lower than the same-day benign median.
- `zero_share`: share of flows where the value is exactly 0 (for `SYN Flag Cnt` this is the share with no SYN flag).
- `single_feature_auc`: how well that one feature alone ranks attack above benign (0.5 = no separation; below 0.5 means attacks are *lower*; far from 0.5 either way means it separates). Computed on samples of up to 100,000 flows per side.

Medians can hide a lot, so read direction and AUC together before writing an assumption as "supported" or "not supported".

In [ ]:
rows = []
for t in ["LOIC-HTTP", "HOIC", "LOIC-UDP"]:
    a = df[df["tool"] == t]
    day = a["source_file"].iloc[0]
    b = df[(df["tool"] == "Benign") & (df["source_file"] == day)]
    s = pd.concat([a.sample(min(len(a), 100_000), random_state=0),
                   b.sample(min(len(b), 100_000), random_state=0)])
    for f in FEATURES + CANDIDATES:   # update 3: candidates 12-15 too
        am, bm = a[f].median(), b[f].median()
        rows.append({
            "attack": t, "feature": f,
            "attack_median": am, "benign_median_same_day": bm,
            "direction": "higher" if am > bm else ("lower" if am < bm else "same"),
            "attack_zero_share": (a[f] == 0).mean(),
            "benign_zero_share": (b[f] == 0).mean(),
            "single_feature_auc": (roc_auc_score(s.loc[s[f].notna(), "y"], s.loc[s[f].notna(), f])
                                   if s.loc[s[f].notna(), "y"].nunique() == 2 else np.nan),
        })
    print(f"{t}: compared with {len(b):,} benign flows from {day}")

evidence = pd.DataFrame(rows)
for t, g in evidence.groupby("attack", sort=False):
    print(f"\n=== {t} vs benign from the same file ===")
    print(g.drop(columns="attack").set_index("feature").round(3).to_string())

print("\nShare of flows with at least one SYN flag, by tool and day:")
print(df.assign(has_syn=df["SYN Flag Cnt"] > 0).groupby(["tool", "source_file"])["has_syn"].mean().round(4))
EVIDENCE_PATH = Path("../DDoSDetector/app/data/feature_evidence.csv")   # read by the desktop app
evidence.to_csv(EVIDENCE_PATH, index=False)
print("\nSaved feature_evidence.csv (use it to fill in the Week 3 feature table).")

## 9c. Do features 8 to 11 help? Same-split comparison

The model uses the original nine columns plus four added feature groups (section 6):
- `reply` (feature 8): `Bwd Pkt Len Mean`, what the victim sends back
- `flags` (feature 9): `RST Flag Cnt`, `FIN Flag Cnt`, how connections end
- `indicators` (feature 10): `no_reply` and `empty_payload`
- `timing` (feature 11): `iat_cv` = IAT std / (IAT mean + 1), how regular the sending is

This cell checks whether they earn their place: the base nine columns, each group added alone, and all 15 together.

**Update 3** adds, on the same splits: all 15 without `SYN Flag Cnt` (14 columns), each candidate group (features 12-15) added alone to those 14, and all candidates together (21 columns). The same rules apply: report every result, and differences smaller than the seed spread are noise.

Fairness rules built into this cell:
- Every feature set is trained and tested on **exactly the same split** (same benign sample, same held-out tool), so differences come from the features only.
- Report every result, including "no improvement". If a group does not help, say so in the write-up and let the group decide whether to keep it. Do not change the feature list quietly because of the held-out tool's score.
- Benign training flows are capped at 300,000 and the model uses 200 trees, so numbers differ from section 7 even for the base set. Compare feature sets with each other, not with section 7.
- Differences smaller than the spread between seeds (`recall_min` to `recall_max`) are noise. The seed here changes the benign sample only.

Set `SEEDS = [0]` for a quick first pass (about a third of the time). Memory note: this cell makes one float32 copy of the needed columns.

In [ ]:
import time

# ---- Feature sets to compare (all columns were built in section 6) ----
ALL_KEY = f"all ({len(FEATURES)} columns)"
FEATURE_SETS = {"base (9 columns)": list(FEATURES_BASE9)}
for name, cols in NEW_GROUPS.items():
    if cols:
        FEATURE_SETS[f"+{name}"] = list(FEATURES_BASE9) + cols
FEATURE_SETS[ALL_KEY] = list(FEATURES)
# Update 3: drop SYN Flag Cnt, then add each candidate group alone, then all candidates.
NO_SYN = [c for c in FEATURES if c != "SYN Flag Cnt"]
FEATURE_SETS[f"all - SYN ({len(NO_SYN)} columns)"] = NO_SYN
for name, cols in CANDIDATE_GROUPS.items():
    if cols:
        FEATURE_SETS[f"- SYN +{name}"] = NO_SYN + cols
FEATURE_SETS[f"- SYN + all candidates ({len(NO_SYN) + len(CANDIDATES)} columns)"] = NO_SYN + CANDIDATES
ALL_USED = list(dict.fromkeys(c for cols in FEATURE_SETS.values() for c in cols))
print("feature sets:", {k: len(v) for k, v in FEATURE_SETS.items()})

# ---- 3. Same-split comparison of the feature sets ----
EXPERIMENTS = [("HOIC", "per_day"), ("HOIC", "by_file"), ("LOIC-HTTP", "per_day")]
SEEDS = [0, 1]
N_TREES = 200
BENIGN_CAP = 300_000

needed = ["source_file", "tool", "y", TIME] + ALL_USED
dd = df.loc[df[TIME].notna(), needed].copy()
for c in ALL_USED:
    dd[c] = dd[c].astype("float32")

def make_split(held_out, benign_mode, seed):
    ben = dd[dd["tool"] == "Benign"]
    att = dd[dd["tool"] != "Benign"]
    if benign_mode == "per_day":
        pct = ben.groupby("source_file")[TIME].rank(pct=True, method="first")
        ben_tr, ben_te = ben[pct <= 0.7], ben[pct > 0.7]
    else:  # by_file: earlier day trains, later day tests
        order = ben.groupby("source_file")[TIME].min().sort_values().index
        ben_tr = ben[ben["source_file"] == order[0]]
        ben_te = ben[ben["source_file"] == order[1]]
    tr = pd.concat([ben_tr.sample(min(len(ben_tr), BENIGN_CAP), random_state=seed),
                    att[att["tool"] != held_out]])
    te = pd.concat([ben_te, att[att["tool"] == held_out]])
    return tr, te

def fit_eval(tr, te, features, seed):
    spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
    m = XGBClassifier(n_estimators=N_TREES, max_depth=6, learning_rate=0.1, tree_method="hist",
                      scale_pos_weight=spw, n_jobs=-1, random_state=seed)
    m.fit(tr[features], tr["y"])
    p = m.predict_proba(te[features])[:, 1]
    y, pred = te["y"].values, p >= 0.5
    return {"recall": pred[y == 1].mean(), "fpr": pred[y == 0].mean(),
            "false_alarms": int(pred[y == 0].sum()),
            "pr_auc": average_precision_score(y, p),
            "median_attack_score": float(np.median(p[y == 1]))}

results = []
for held, mode in EXPERIMENTS:
    for seed in SEEDS:
        tr, te = make_split(held, mode, seed)
        for name, feats in FEATURE_SETS.items():
            t0 = time.perf_counter()
            r = fit_eval(tr, te, feats, seed)
            r.update(held_out=held, benign_mode=mode, seed=seed, feature_set=name, n_features=len(feats))
            results.append(r)
            print(f"{held}/{mode} seed {seed} {name}: recall {r['recall']:.3f}, "
                  f"FPR {r['fpr']:.4f} ({time.perf_counter() - t0:.0f}s)")

res = pd.DataFrame(results)
summary = (res.groupby(["held_out", "benign_mode", "feature_set"], sort=False)
             .agg(recall=("recall", "mean"), recall_min=("recall", "min"), recall_max=("recall", "max"),
                  fpr=("fpr", "mean"), false_alarms=("false_alarms", "mean"),
                  pr_auc=("pr_auc", "mean"), median_attack_score=("median_attack_score", "mean")))
print("\n=== Held-out results by feature set (threshold 0.5, mean over seeds) ===")
print(summary.round(4).to_string())
COMPARISON_PATH = Path("../DDoSDetector/models/feature_set_comparison.csv")   # read by the desktop app
res.to_csv(COMPARISON_PATH, index=False)
print("\nSaved feature_set_comparison.csv")

## 10. In-distribution baseline

Random split with all tools seen in training. This is an **upper bound**: flows from the same attack burst appear in both train and test, so near-duplicates inflate it. Comparing it with section 7 separates "the features cannot detect this attack" from "the model cannot transfer to an unseen tool".

In [ ]:
def baseline(seed=0, benign_n=1_000_000):
    ben = df[df["tool"] == "Benign"].sample(benign_n, random_state=seed)
    d = pd.concat([ben, df[df["tool"] != "Benign"]])
    tr, te = train_test_split(d, test_size=0.3, stratify=d["tool"], random_state=seed)
    spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
    model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist",
                          scale_pos_weight=spw, n_jobs=-1, random_state=seed).fit(tr[FEATURES], tr["y"])
    return model, te

mb, teb = baseline(seed=0)
pred = mb.predict_proba(teb[FEATURES])[:, 1] >= 0.5
yb = teb["y"].values
print("baseline FPR:", round(pred[yb == 0].mean(), 4), f"({int(pred[yb == 0].sum()):,} false alarms)")
for t in ["HOIC", "LOIC-HTTP", "LOIC-UDP"]:
    m = (teb["tool"] == t).values
    print(f"baseline recall {t}: {pred[m].mean():.3f} (n={m.sum():,})")

## 11. SHAP explanations (not yet run: check it works before relying on it)

Run `pip install shap` once if the import fails. Part A shows which features drive the baseline model's decisions for each tool (mean absolute SHAP value). Part B retrains the baseline with several seeds and checks whether the feature ranking is stable (the Week 4 stability check). Benign flows are capped at 300,000 per seed here to keep it quick.

In [ ]:
import shap

def mean_abs_shap(model, te, per_tool=2000, seed=0):
    explainer = shap.TreeExplainer(model)
    out = {}
    for t in ["Benign", "HOIC", "LOIC-HTTP", "LOIC-UDP"]:
        s = te[te["tool"] == t]
        s = s.sample(min(per_tool, len(s)), random_state=seed)
        sv = explainer.shap_values(s[FEATURES])
        out[t] = np.abs(sv).mean(axis=0)
    return pd.DataFrame(out, index=FEATURES)

# Part A: which features matter for which tool (baseline model from section 10)
imp = mean_abs_shap(mb, teb)
print(imp.round(3))

In [ ]:
# Part B: stability across random seeds (attack tools averaged)
SEEDS = [0, 1, 2, 3, 4]
per_seed = {}
for s in SEEDS:
    m_s, te_s = baseline(seed=s, benign_n=300_000)
    imp_s = mean_abs_shap(m_s, te_s, seed=s)
    per_seed[s] = imp_s[["HOIC", "LOIC-HTTP", "LOIC-UDP"]].mean(axis=1)
    print("seed", s, "top 3:", list(per_seed[s].sort_values(ascending=False).index[:3]))

stab = pd.DataFrame(per_seed)
print(stab.round(3))
print("Spearman rank correlation between seeds:")
print(stab.corr(method="spearman").round(2))

## 12. Findings so far (for the Week 4 write-up)

- **Important:** the results below were obtained with the original **nine** input columns. The model now uses 15 columns (features 8 to 11 added), so re-run sections 7, 8, 10 and 11 and update these numbers; section 9c shows the nine-column result next to the 15-column result on identical splits.
- **Cleaning:** 8,997,323 -> 8,872,017 rows (1.4% removed). LOIC-UDP unchanged at 1,730.
- **Baseline, all tools seen in training:** recall HOIC 0.999, LOIC-HTTP 0.987, LOIC-UDP 1.000 (519 test flows), FPR 0.0084. Treat as an upper bound.
- **Held-out HOIC (train LOIC-HTTP + LOIC-UDP):** recall 0.156 (per_day) / 0.195 (by_file) at threshold 0.5; at most about 0.40 at any threshold tried.
- **Held-out LOIC-HTTP:** recall 0.003, FPR 0.0054.
- **Interpretation:** the features can detect each tool when the model has seen it, but the model does not transfer to an unseen tool. The three tools differ by orders of magnitude in duration, packet rate and payload, so two tools give no shared signature to learn.
- **Limitations:** feature 7 dropped (no `Dst IP` on Wednesday); LOIC-UDP is tiny; tool and day are confounded; random-split baseline is optimistic; `SYN Flag Cnt` had median 0 for every tool and no gain in the held-out model.